# some TESLA data prelim analysis

In [ ]:
import polars as pl

from as_4777_conformance import volt_watt_conformance, passive_anti_islanding
from helper_funcs import estimate_rated_capacity

In [ ]:
site_metadata = pl.read_excel("dataset/SITE_METADATA.xlsx")

In [ ]:
more_than_one_inverter = site_metadata.filter(pl.col('pw3_inverters')!=1)
more_than_one_inverter

In [ ]:
# separate dc vs ac coupled systems
dc_systems         = site_metadata.filter((pl.col('solar_coupling')=='dc_only') & (pl.col('pw3_inverters')==1))
ac_coupled_systems = site_metadata.filter((pl.col('solar_coupling')=='ac_coupled') & (pl.col('pw3_inverters')==1))

In [ ]:
site_capacity_rows = []

for site in pl.concat([dc_systems, ac_coupled_systems]).iter_rows(named=True):
    telemetry_path = f"dataset/{site['folder']}/{site['pseudonym']}_telemetry_60sec.parquet"
    site_telemetry = pl.read_parquet(
        telemetry_path,
        columns=[
            'inverter_ac_real_power',
            'inverter_ac_reactive_power',
        ],
    )
    metadata_capacity, s99_capacity, calc_capacity = estimate_rated_capacity(
        site['system_power_kw_ac'],
        site_telemetry,
    )
    site_capacity_rows.append({
        'pseudonym': site['pseudonym'],
        'metadata_capacity': metadata_capacity,
        's99_capacity': s99_capacity,
        'calc_capacity': calc_capacity,
    })

site_capacity = pl.DataFrame(site_capacity_rows)
dc_systems = dc_systems.join(site_capacity, on='pseudonym', how='left')
ac_coupled_systems = ac_coupled_systems.join(
    site_capacity, on='pseudonym', how='left'
)

site_capacity

In [ ]:
len(ac_coupled_systems)

# DC onnly system analysis

In [ ]:
dc_systems[0]

In [ ]:
# data coverage
# for SAPN
# for AUSNET
# for CPU

In [ ]:
# investigate a dc_only site for pv_inverter/ battery/ solar behavior
test_site = dc_systems.row(0, named=True)
telemetry_path = f"dataset/{test_site['folder']}/{test_site['pseudonym']}_telemetry_60sec.parquet"
test_site_telemetry = pl.read_parquet(telemetry_path)
test_site_telemetry

In [ ]:
test_site_telemetry.columns

In [ ]:
pl.Config.set_tbl_rows(20)
# verify inverter additive output
# battery_power: Battery charge (−) / discharge (+)
total_power_output = test_site_telemetry['pw_dc_pv_power'] + test_site_telemetry['battery_power']
inv_power_diff     = ((total_power_output - \
                        test_site_telemetry['inverter_ac_real_power']).drop_nulls())/1000
# inv_power_diff.abs().sort(descending=True)

In [ ]:
# import matplotlib.pyplot as plt

# plt.figure(figsize=(8, 5))
# plt.hist(inv_power_diff.to_numpy(), bins=100, edgecolor="black")
# plt.axvline(0, color="red", linestyle="--", linewidth=1)
# plt.xlabel("Residual: DC PV + battery - inverter AC output (kW)")
# plt.ylabel("Frequency")
# plt.title("Inverter residual distribution\n(DC PV + battery - inverter AC output)")
# plt.tight_layout()
# plt.show()

In [ ]:
test_site_inv_capacity = test_site['calc_capacity']
test_site_inv_tol      = test_site_inv_capacity * 0.04

num_timestamps = test_site_telemetry.height
num_non_null_timestamps = inv_power_diff.len()
percentage_within_tolerance = (inv_power_diff.abs() <= test_site_inv_tol).mean() * 100

print(f"Number of timestamps: {num_timestamps}")
print(f"Number of non-null timestamps: {num_non_null_timestamps}")
print(f"Percentage within tolerance: {percentage_within_tolerance:.2f}%")
print(f"Max absolute inv_power_diff: {inv_power_diff.abs().max()}")

In [ ]:
# verify inverter output for the whole dc_systems fleet
# battery_power: Battery charge (−) / discharge (+)
fleet_inv_power_diff = []
fleet_within_tolerance = []
site_tolerance_summary = []
fleet_num_timestamps = 0

for site in dc_systems.iter_rows(named=True):
    telemetry_path = f"dataset/{site['folder']}/{site['pseudonym']}_telemetry_60sec.parquet"
    site_telemetry = pl.read_parquet(telemetry_path)

    total_power_output = site_telemetry['pw_dc_pv_power'] + site_telemetry['battery_power']
    site_inv_power_diff = ((total_power_output - \
                            site_telemetry['inverter_ac_real_power']).drop_nulls())/1000
    site_inv_tol = site['calc_capacity'] * 0.04

    fleet_inv_power_diff.append(site_inv_power_diff)
    fleet_within_tolerance.append(site_inv_power_diff.abs() <= site_inv_tol)
    site_tolerance_summary.append({
        'pseudonym': site['pseudonym'],
        'percentage_within_tolerance': (site_inv_power_diff.abs() <= site_inv_tol).mean() * 100,
        'num_non_null_timestamps': site_inv_power_diff.len(),
    })
    fleet_num_timestamps += site_telemetry.height

fleet_inv_power_diff = pl.concat(fleet_inv_power_diff)
fleet_within_tolerance = pl.concat(fleet_within_tolerance)
site_tolerance_summary = pl.DataFrame(site_tolerance_summary)
# fleet_inv_power_diff.abs().sort(descending=True)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.hist(fleet_inv_power_diff.to_numpy(), bins=100, edgecolor="black")
plt.axvline(0, color="red", linestyle="--", linewidth=1)
plt.xlabel("Residual: DC PV + battery - inverter AC output (kW)")
plt.ylabel("Frequency")
plt.title("DC fleet distribution\n(DC PV + battery - inverter AC output)")
plt.tight_layout()
plt.show()

In [ ]:
num_timestamps = fleet_num_timestamps
num_non_null_timestamps = fleet_inv_power_diff.len()
percentage_within_tolerance = fleet_within_tolerance.mean() * 100

print(f"Number of DC systems: {len(dc_systems)}")
print(f"Number of timestamps: {num_timestamps}")
print(f"Number of non-null timestamps: {num_non_null_timestamps}")
print(f"Percentage within tolerance: {percentage_within_tolerance:.2f}%")
print(f"Max absolute inv_power_diff: {fleet_inv_power_diff.abs().max()}")

In [ ]:
site_tolerance_plot = site_tolerance_summary.sort('percentage_within_tolerance')
fleet_percentage_within_tolerance = fleet_within_tolerance.mean() * 100

plt.figure(figsize=(18, 6))
plt.bar(
    site_tolerance_plot['pseudonym'].to_list(),
    site_tolerance_plot['percentage_within_tolerance'].to_list(),
    edgecolor='black',
)
plt.axhline(
    fleet_percentage_within_tolerance,
    color='red',
    linestyle='--',
    linewidth=1,
    label=f'Fleet: {fleet_percentage_within_tolerance:.2f}%',
)
plt.ylim(0, 100)
plt.yticks(range(0, 101, 5))
plt.xlabel('Site ID', fontsize=14)
plt.ylabel('Timestamps within tolerance (%)', fontsize=14)
plt.xticks(rotation=90)
plt.title(
    'DC sites: percentage of timestamps where |DC PV + battery - inverter AC output|\n'
    '<= 4% of rated inverter capacity',
    fontsize=16,
)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# verifyung solar values at DC bus on inverter and site solar at the meter
solar_difference       =  test_site_telemetry['pw_dc_pv_power'] - test_site_telemetry['site_solar_power']
# Summing a Boolean mask counts its True values (timestamps exceeding the tolerance).
exceeding_tol          = (solar_difference
                          / 1000 > test_site_inv_tol)
exceeding_tol_values   = solar_difference.filter(exceeding_tol)/1000
# exceeding_tol_values.sort(descending=True)

In [ ]:
exceeding_tol_counts   = exceeding_tol.sum()
exceeding_tol_counts

In [ ]:
num_solar_timestamps = test_site_telemetry.height
num_non_null_solar_timestamps = solar_difference.drop_nulls().len()
percentage_within_solar_tolerance = (~exceeding_tol).mean() * 100

print(f"Number of timestamps: {num_solar_timestamps}")
print(f"Number of non-null timestamps: {num_non_null_solar_timestamps}")
print(f"Percentage within tolerance: {percentage_within_solar_tolerance:.2f}%")
print(f"Max absolute exceeding_tol_values: {exceeding_tol_values.abs().max()}")

In [ ]:
# Verify solar behaviour for the whole dc_systems fleet
fleet_solar_difference = []
fleet_exceeding_tol = []
fleet_exceeding_tol_values = []
site_solar_tolerance_summary = []
fleet_num_solar_timestamps = 0

for site in dc_systems.iter_rows(named=True):
    telemetry_path = f"dataset/{site['folder']}/{site['pseudonym']}_telemetry_60sec.parquet"
    site_telemetry = pl.read_parquet(telemetry_path)

    site_solar_difference = site_telemetry['pw_dc_pv_power'] - site_telemetry['site_solar_power']
    site_inv_tol = site['calc_capacity'] * 0.04
    # Summing a Boolean mask counts its True values (timestamps exceeding the tolerance).
    site_exceeding_tol = (site_solar_difference
                          / 1000 > site_inv_tol)
    site_exceeding_tol_values = site_solar_difference.filter(site_exceeding_tol)/1000

    fleet_solar_difference.append(site_solar_difference)
    fleet_exceeding_tol.append(site_exceeding_tol)
    fleet_exceeding_tol_values.append(site_exceeding_tol_values)
    site_solar_tolerance_summary.append({
        'pseudonym': site['pseudonym'],
        'percentage_within_tolerance': (~site_exceeding_tol).mean() * 100,
        'num_non_null_timestamps': site_solar_difference.drop_nulls().len(),
    })
    fleet_num_solar_timestamps += site_telemetry.height

fleet_solar_difference = pl.concat(fleet_solar_difference)
fleet_exceeding_tol = pl.concat(fleet_exceeding_tol)
fleet_exceeding_tol_values = pl.concat(fleet_exceeding_tol_values)
site_solar_tolerance_summary = pl.DataFrame(site_solar_tolerance_summary)
# fleet_exceeding_tol_values.sort(descending=True)

In [ ]:
fleet_exceeding_tol_counts = fleet_exceeding_tol.sum()
fleet_exceeding_tol_counts

In [ ]:
num_solar_timestamps = fleet_num_solar_timestamps
num_non_null_solar_timestamps = fleet_solar_difference.drop_nulls().len()
percentage_within_solar_tolerance = (~fleet_exceeding_tol).mean() * 100

print(f"Number of DC systems: {len(dc_systems)}")
print(f"Number of timestamps: {num_solar_timestamps}")
print(f"Number of non-null timestamps: {num_non_null_solar_timestamps}")
print(f"Percentage within tolerance: {percentage_within_solar_tolerance:.2f}%")
print(f"Max absolute exceeding_tol_values: {fleet_exceeding_tol_values.abs().max()}")

In [ ]:
site_solar_tolerance_plot = site_solar_tolerance_summary.sort('percentage_within_tolerance')

plt.figure(figsize=(18, 6))
plt.bar(
    site_solar_tolerance_plot['pseudonym'].to_list(),
    site_solar_tolerance_plot['percentage_within_tolerance'].to_list(),
    edgecolor='black',
)
plt.axhline(
    percentage_within_solar_tolerance,
    color='red',
    linestyle='--',
    linewidth=1,
    label=f'Fleet: {percentage_within_solar_tolerance:.2f}%',
)
plt.ylim(0, 100)
plt.yticks(range(0, 101, 5))
plt.xlabel('Site ID', fontsize=14)
plt.ylabel('Timestamps within tolerance (%)', fontsize=14)
plt.xticks(rotation=90)
plt.title(
    'DC sites: percentage of timestamps where inverter solar and POC solar differ\n'
    'by no more than 4% of rated inverter capacity',
    fontsize=16,
)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Preliminary CSIP analysis for the DC fleet
# site_instant_power: import (+) / export (-)

csip_dc_systems = dc_systems.filter(pl.col('pseudonym') != 'SN100') # exclude site SN100 as it does not have vlaid CSIP signals

fleet_csip_tolerance_exceedance_w = []
site_csip_compliance_summary = []
dc_volt_watt_conformance_summary = []
fleet_num_csip_timestamps = 0

for site in dc_systems.iter_rows(named=True):
    telemetry_path = f"dataset/{site['folder']}/{site['pseudonym']}_telemetry_60sec.parquet"
    site_telemetry = pl.read_parquet(telemetry_path).sort('timestamp')

    # The CSIP limit is a step signal reported on change and as a daily heartbeat.
    # Forward filling leaves timestamps before the first reported limit null.
    site_telemetry = site_telemetry.with_columns(
        pl.col('csip_export_limit_w')
        .forward_fill()
        .alias('csip_export_limit_w_ffill')
    )

    site_tolerance_w = site['calc_capacity'] * 1000 * 0.04
    site_inv_power_diff_w = (
        site_telemetry['pw_dc_pv_power']
        + site_telemetry['battery_power']
        - site_telemetry['inverter_ac_real_power']
    )
    site_solar_difference_w = (
        site_telemetry['pw_dc_pv_power']
        - site_telemetry['site_solar_power']
    )

    site_valid_volt_watt_mask = (
        site_inv_power_diff_w.is_not_null()
        & site_solar_difference_w.is_not_null()
        & site_telemetry['grid_voltage'].is_not_null()
        & site_telemetry['inverter_ac_real_power'].is_not_null()
        & (site_inv_power_diff_w.abs() <= site_tolerance_w)
        & (site_solar_difference_w.abs() <= site_tolerance_w)
    )
    site_volt_watt_result = volt_watt_conformance(
        site_telemetry.filter(site_valid_volt_watt_mask),
        site['calc_capacity'],
        min_eligible_timestamps=20,
    ).select(pl.lit(site['pseudonym']).alias('pseudonym'), pl.all())
    dc_volt_watt_conformance_summary.append(site_volt_watt_result)

    if site['pseudonym'] == 'SN100':
        continue

    # filter out timestamps with valid values
    site_valid_csip_mask = (
        site_inv_power_diff_w.is_not_null()
        & site_solar_difference_w.is_not_null()
        & site_telemetry['site_instant_power'].is_not_null()
        & site_telemetry['csip_export_limit_w_ffill'].is_not_null()
        & (site_inv_power_diff_w.abs() <= site_tolerance_w)
        & (site_solar_difference_w.abs() <= site_tolerance_w)
    )

    # CSIP export check: export_W = max(0, −site_instant_power)
    # - sign introduced becasue export to gtid is negative
    site_export_w = (
        -site_telemetry['site_instant_power']
    ).clip(lower_bound=0).filter(site_valid_csip_mask) # clipping to remove importing value. But if you dont do it, the results should be the same
    site_export_limit_w = site_telemetry[
        'csip_export_limit_w_ffill'
    ].filter(site_valid_csip_mask)
    # Positive values exceed the CSIP limit plus the 4% capacity allowance.
    site_csip_tolerance_exceedance_w = (
        site_export_w - site_export_limit_w - site_tolerance_w
    ).rename('csip_tolerance_exceedance_w')
    site_csip_within_tolerance = site_csip_tolerance_exceedance_w <= 0

    fleet_csip_tolerance_exceedance_w.append(site_csip_tolerance_exceedance_w)
    site_csip_compliance_summary.append({
        'pseudonym': site['pseudonym'],
        'percentage_within_tolerance': site_csip_within_tolerance.mean() * 100,
        'num_non_null_timestamps': site_csip_tolerance_exceedance_w.len(),
        'max_export_exceedance_w': site_csip_tolerance_exceedance_w.clip(lower_bound=0).max(),
    })
    fleet_num_csip_timestamps += site_telemetry.height

dc_volt_watt_conformance_summary = pl.concat(dc_volt_watt_conformance_summary)

fleet_csip_tolerance_exceedance_w = pl.concat(fleet_csip_tolerance_exceedance_w)
fleet_csip_within_tolerance = fleet_csip_tolerance_exceedance_w <= 0
site_csip_compliance_summary = pl.DataFrame(site_csip_compliance_summary)

num_csip_timestamps = fleet_num_csip_timestamps
num_non_null_csip_timestamps = fleet_csip_tolerance_exceedance_w.len()
percentage_within_csip_tolerance = fleet_csip_within_tolerance.mean() * 100
max_csip_export_exceedance_w = (
    fleet_csip_tolerance_exceedance_w.clip(lower_bound=0).max()
)

print(f"Number of DC systems: {len(csip_dc_systems)} (SN100 excluded)")
print(f"Number of timestamps: {num_csip_timestamps}")
print(f"Number of eligible timestamps: {num_non_null_csip_timestamps}")
print(f"Percentage within CSIP tolerance: {percentage_within_csip_tolerance:.2f}%")
print(f"Max export exceedance beyond tolerance: {max_csip_export_exceedance_w:.2f} W")
# site_csip_compliance_summary['percentage_within_tolerance'].min()
# display(site_csip_compliance_summary)
# dc_volt_watt_conformance_summary

In [ ]:
site_csip_compliance_plot = site_csip_compliance_summary.sort(
    'percentage_within_tolerance'
)
site_labels = site_csip_compliance_plot['pseudonym'].to_list()
x_positions = range(len(site_labels))

fig, compliance_ax = plt.subplots(figsize=(18, 6))
compliance_ax.bar(
    x_positions,
    site_csip_compliance_plot['percentage_within_tolerance'].to_list(),
    edgecolor='black',
)
fleet_line = compliance_ax.axhline(
    percentage_within_csip_tolerance,
    color='red',
    linestyle='--',
    linewidth=2.5,
    label=f'Fleet: {percentage_within_csip_tolerance:.2f}%',
)
compliance_ax.set_ylim(0, 100)
compliance_ax.set_yticks(range(0, 101, 5))
compliance_ax.set_xlabel('Site ID', fontsize=14)
compliance_ax.set_ylabel('Timestamps within CSIP tolerance (%)', fontsize=14)
compliance_ax.set_xticks(x_positions)
compliance_ax.set_xticklabels(site_labels, rotation=90)

timestamp_ax = compliance_ax.twinx()
timestamp_line, = timestamp_ax.plot(
    x_positions,
    site_csip_compliance_plot['num_non_null_timestamps'].to_list(),
    color='darkorange',
    marker='o',
    markersize=4,
    linewidth=2.5,
    label='Valid timestamps',
)
timestamp_ax.set_ylabel(
    'Number of valid timestamps', color='darkorange', fontsize=14
)
timestamp_ax.tick_params(axis='y', labelcolor='darkorange')

compliance_ax.set_title(
    'CSIP compliance and valid timestamps by site', fontsize=16
)
compliance_ax.legend(handles=[fleet_line, timestamp_line], loc='lower right')
fig.tight_layout()
plt.show()

# AC fleet analysis

In [ ]:
# verify inverter output for the whole ac_coupled_systems fleet
# battery_power: Battery charge (−) / discharge (+)
ac_fleet_inv_power_diff = []
ac_fleet_within_tolerance = []
ac_site_tolerance_summary = []
ac_fleet_num_timestamps = 0

for site in ac_coupled_systems.iter_rows(named=True):
    telemetry_path = f"dataset/{site['folder']}/{site['pseudonym']}_telemetry_60sec.parquet"
    site_telemetry = pl.read_parquet(telemetry_path)

    total_power_output = site_telemetry['pw_dc_pv_power'] + site_telemetry['battery_power']
    site_inv_power_diff = ((total_power_output - \
                            site_telemetry['inverter_ac_real_power']).drop_nulls())/1000
    site_inv_tol = site['calc_capacity'] * 0.04

    ac_fleet_inv_power_diff.append(site_inv_power_diff)
    ac_fleet_within_tolerance.append(site_inv_power_diff.abs() <= site_inv_tol)
    ac_site_tolerance_summary.append({
        'pseudonym': site['pseudonym'],
        'percentage_within_tolerance': (site_inv_power_diff.abs() <= site_inv_tol).mean() * 100,
        'num_non_null_timestamps': site_inv_power_diff.len(),
    })
    ac_fleet_num_timestamps += site_telemetry.height

ac_fleet_inv_power_diff = pl.concat(ac_fleet_inv_power_diff)
ac_fleet_within_tolerance = pl.concat(ac_fleet_within_tolerance)
ac_site_tolerance_summary = pl.DataFrame(ac_site_tolerance_summary)
# ac_fleet_inv_power_diff.abs().sort(descending=True)

In [ ]:
ac_num_timestamps = ac_fleet_num_timestamps
ac_num_non_null_timestamps = ac_fleet_inv_power_diff.len()
ac_percentage_within_tolerance = ac_fleet_within_tolerance.mean() * 100

print(f"Number of AC systems: {len(ac_coupled_systems)}")
print(f"Number of timestamps: {ac_num_timestamps}")
print(f"Number of non-null timestamps: {ac_num_non_null_timestamps}")
print(f"Percentage within tolerance: {ac_percentage_within_tolerance:.2f}%")
print(f"Max absolute inv_power_diff: {ac_fleet_inv_power_diff.abs().max()}")

In [ ]:
ac_site_tolerance_plot = ac_site_tolerance_summary.sort('percentage_within_tolerance')
ac_fleet_percentage_within_tolerance = ac_fleet_within_tolerance.mean() * 100

plt.figure(figsize=(18, 6))
plt.bar(
    ac_site_tolerance_plot['pseudonym'].to_list(),
    ac_site_tolerance_plot['percentage_within_tolerance'].to_list(),
    edgecolor='black',
)
plt.axhline(
    ac_fleet_percentage_within_tolerance,
    color='red',
    linestyle='--',
    linewidth=1,
    label=f'Fleet: {ac_fleet_percentage_within_tolerance:.2f}%',
)
plt.ylim(0, 100)
plt.yticks(range(0, 101, 5))
plt.xlabel('Site ID', fontsize=14)
plt.ylabel('Timestamps within tolerance (%)', fontsize=14)
plt.xticks(rotation=90)
plt.title(
    'AC-coupled sites: percentage of timestamps where |DC PV + battery - inverter AC output|\n'
    '<= 4% of rated inverter capacity (external AC solar excluded)',
    fontsize=16,
)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Verify solar behaviour for the whole AC-coupled fleet
# AC-coupled solar can make site_solar_power greater than pw_dc_pv_power,
# so only positive differences beyond the tolerance are treated as failures.
ac_fleet_solar_difference = []
ac_fleet_solar_exceeding_tol = []
ac_fleet_solar_exceeding_tol_values = []
ac_site_solar_tolerance_summary = []
ac_fleet_num_solar_timestamps = 0

for site in ac_coupled_systems.iter_rows(named=True):
    telemetry_path = f"dataset/{site['folder']}/{site['pseudonym']}_telemetry_60sec.parquet"
    site_telemetry = pl.read_parquet(telemetry_path)

    site_solar_difference = (
        site_telemetry['pw_dc_pv_power']
        - site_telemetry['site_solar_power']
    ) / 1000
    site_solar_tolerance = site['calc_capacity'] * 0.04
    site_solar_exceeding_tol = (
        site_solar_difference > site_solar_tolerance
    )
    site_solar_exceeding_tol_values = site_solar_difference.filter(
        site_solar_exceeding_tol
    )

    ac_fleet_solar_difference.append(site_solar_difference)
    ac_fleet_solar_exceeding_tol.append(site_solar_exceeding_tol)
    ac_fleet_solar_exceeding_tol_values.append(
        site_solar_exceeding_tol_values
    )
    ac_site_solar_tolerance_summary.append({
        'pseudonym': site['pseudonym'],
        'percentage_within_tolerance': (~site_solar_exceeding_tol).mean() * 100,
        'num_non_null_timestamps': site_solar_difference.drop_nulls().len(),
    })
    ac_fleet_num_solar_timestamps += site_telemetry.height

ac_fleet_solar_difference = pl.concat(ac_fleet_solar_difference)
ac_fleet_solar_exceeding_tol = pl.concat(ac_fleet_solar_exceeding_tol)
ac_fleet_solar_exceeding_tol_values = pl.concat(
    ac_fleet_solar_exceeding_tol_values
)
ac_site_solar_tolerance_summary = pl.DataFrame(
    ac_site_solar_tolerance_summary
)

In [ ]:
ac_fleet_solar_exceeding_tol_count = ac_fleet_solar_exceeding_tol.sum()
ac_fleet_solar_exceeding_tol_count

In [ ]:
ac_num_solar_timestamps = ac_fleet_num_solar_timestamps
ac_num_non_null_solar_timestamps = (
    ac_fleet_solar_difference.drop_nulls().len()
)
ac_percentage_within_solar_tolerance = (
    (~ac_fleet_solar_exceeding_tol).mean() * 100
)

print(f"Number of AC systems: {len(ac_coupled_systems)}")
print(f"Number of timestamps: {ac_num_solar_timestamps}")
print(f"Number of non-null timestamps: {ac_num_non_null_solar_timestamps}")
print(f"Percentage within tolerance: {ac_percentage_within_solar_tolerance:.2f}%")
print(
    "Max solar difference exceeding tolerance: "
    f"{ac_fleet_solar_exceeding_tol_values.max()} kW"
)

In [ ]:
ac_site_solar_tolerance_plot = ac_site_solar_tolerance_summary.sort(
    'percentage_within_tolerance'
)

plt.figure(figsize=(18, 6))
plt.bar(
    ac_site_solar_tolerance_plot['pseudonym'].to_list(),
    ac_site_solar_tolerance_plot['percentage_within_tolerance'].to_list(),
    edgecolor='black',
)
plt.axhline(
    ac_percentage_within_solar_tolerance,
    color='red',
    linestyle='--',
    linewidth=1,
    label=f'Fleet: {ac_percentage_within_solar_tolerance:.2f}%',
)
plt.ylim(0, 100)
plt.yticks(range(0, 101, 5))
plt.xlabel('Site ID', fontsize=14)
plt.ylabel('Timestamps within tolerance (%)', fontsize=14)
plt.xticks(rotation=90)
plt.title(
    'AC-coupled sites: percentage of timestamps where inverter solar does not exceed POC solar\n'
    'by more than 4% of rated inverter capacity',
    fontsize=16,
)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Volt-Watt conformance analysis for the AC-coupled fleet
ac_volt_watt_conformance_summary = []

for site in ac_coupled_systems.iter_rows(named=True):
    telemetry_path = f"dataset/{site['folder']}/{site['pseudonym']}_telemetry_60sec.parquet"
    site_telemetry = pl.read_parquet(telemetry_path).sort('timestamp')

    site_tolerance_w = site['calc_capacity'] * 1000 * 0.04
    site_inv_power_diff_w = (
        site_telemetry['pw_dc_pv_power']
        + site_telemetry['battery_power']
        - site_telemetry['inverter_ac_real_power']
    )
    site_solar_difference_w = (
        site_telemetry['pw_dc_pv_power']
        - site_telemetry['site_solar_power']
    )

    site_valid_volt_watt_mask = (
        site_inv_power_diff_w.is_not_null()
        & site_solar_difference_w.is_not_null()
        & site_telemetry['grid_voltage'].is_not_null()
        & site_telemetry['inverter_ac_real_power'].is_not_null()
        & (site_inv_power_diff_w.abs() <= site_tolerance_w)
        & (site_solar_difference_w <= site_tolerance_w)
    )
    site_volt_watt_result = volt_watt_conformance(
        site_telemetry.filter(site_valid_volt_watt_mask),
        site['calc_capacity'],
        min_eligible_timestamps=20,
    ).select(pl.lit(site['pseudonym']).alias('pseudonym'), pl.all())
    ac_volt_watt_conformance_summary.append(site_volt_watt_result)

ac_volt_watt_conformance_summary = pl.concat(
    ac_volt_watt_conformance_summary
)
# ac_volt_watt_conformance_summary

In [ ]:
volt_watt_conformance_summary = pl.concat(
    [
        dc_volt_watt_conformance_summary,
        ac_volt_watt_conformance_summary,
    ]
)

volt_watt_conformance_plot = (
    volt_watt_conformance_summary
    .filter(pl.col('num_eligible_timestamps') >= 20)
    .sort('conformant_percentage')
)
site_labels = volt_watt_conformance_plot['pseudonym'].to_list()
x_positions = range(len(site_labels))
fleet_conformant_percentage = (
    volt_watt_conformance_plot['num_conformant_timestamps'].sum()
    / volt_watt_conformance_plot['num_eligible_timestamps'].sum()
    * 100
)

fig, conformance_ax = plt.subplots(figsize=(18, 6))
conformance_ax.bar(
    x_positions,
    volt_watt_conformance_plot['conformant_percentage'].to_list(),
    edgecolor='black',
)
threshold_line = conformance_ax.axhline(
    90,
    color='green',
    linestyle='--',
    linewidth=2.5,
    label='Conformance threshold: 90%',
)
fleet_line = conformance_ax.axhline(
    fleet_conformant_percentage,
    color='red',
    linestyle='--',
    linewidth=2.5,
    label=f'Fleet: {fleet_conformant_percentage:.2f}%',
)
conformance_ax.set_ylim(0, 100)
conformance_ax.set_yticks(range(0, 101, 5))
conformance_ax.set_xlabel('Site ID', fontsize=14)
conformance_ax.set_ylabel('Volt-Watt conformant timestamps (%)', fontsize=14)
conformance_ax.set_xticks(x_positions)
conformance_ax.set_xticklabels(site_labels, rotation=90)

timestamp_ax = conformance_ax.twinx()
timestamp_line, = timestamp_ax.plot(
    x_positions,
    volt_watt_conformance_plot['num_eligible_timestamps'].to_list(),
    color='darkorange',
    marker='o',
    markersize=4,
    linewidth=2.5,
    label='Eligible timestamps',
)
timestamp_ax.set_ylabel(
    'Number of eligible timestamps', color='darkorange', fontsize=14
)
timestamp_ax.tick_params(axis='y', labelcolor='darkorange')

conformance_ax.set_title(
    'Volt-Watt conformance and eligible timestamps by site', fontsize=16
)
conformance_ax.legend(
    handles=[threshold_line, fleet_line, timestamp_line], loc='lower right'
)
fig.tight_layout()
plt.show()


In [ ]:
volt_watt_conformance_summary

In [ ]:
num_sites_investigated = volt_watt_conformance_summary.height
num_sites_eligible = volt_watt_conformance_plot.height
num_sites_conformant = volt_watt_conformance_plot.filter(
    pl.col('conformant_percentage') >= 90
).height
conformant_percentage_of_cohort = (
    num_sites_conformant / num_sites_eligible * 100
)

print(f'Number of sites investigated: {num_sites_investigated}')
print(f'Number of sites eligible: {num_sites_eligible}')
print(f'Number of sites conformant: {num_sites_conformant}')
print(
    f'Conformant percentage of cohort: '
    f'{conformant_percentage_of_cohort:.2f}%'
)
                           

In [ ]:
all_single_inverter_systems = pl.concat(
    [dc_systems, ac_coupled_systems]
)
passive_anti_islanding_conformance_summary = []

for site in all_single_inverter_systems.iter_rows(named=True):
    telemetry_path = f"dataset/{site['folder']}/{site['pseudonym']}_telemetry_60sec.parquet"
    site_telemetry = pl.read_parquet(telemetry_path).sort('timestamp')

    site_passive_anti_islanding_result = passive_anti_islanding(
        site_telemetry,
        site['calc_capacity'],
        min_eligible_timestamps=20,
    ).select(
        pl.lit(site['pseudonym']).alias('pseudonym'),
        pl.all(),
        pl.lit(site['solar_coupling']).alias('solar_coupling'),
    )
    passive_anti_islanding_conformance_summary.append(
        site_passive_anti_islanding_result
    )

passive_anti_islanding_conformance_summary = pl.concat(
    passive_anti_islanding_conformance_summary
)
passive_anti_islanding_conformance_summary

In [ ]:
passive_anti_islanding_conformance_plot = (
    passive_anti_islanding_conformance_summary
    .filter(pl.col('num_eligible_timestamps') >= 20)
    .sort('conformant_percentage')
)
site_labels = (
    passive_anti_islanding_conformance_plot['pseudonym'].to_list()
)
x_positions = range(len(site_labels))
fleet_conformant_percentage = (
    passive_anti_islanding_conformance_plot[
        'num_conformant_timestamps'
    ].sum()
    / passive_anti_islanding_conformance_plot[
        'num_eligible_timestamps'
    ].sum()
    * 100
)

fig, conformance_ax = plt.subplots(figsize=(18, 6))
conformance_ax.bar(
    x_positions,
    passive_anti_islanding_conformance_plot[
        'conformant_percentage'
    ].to_list(),
    edgecolor='black',
)
threshold_line = conformance_ax.axhline(
    90,
    color='green',
    linestyle='--',
    linewidth=2.5,
    label='Conformance threshold: 90%',
)
fleet_line = conformance_ax.axhline(
    fleet_conformant_percentage,
    color='red',
    linestyle='--',
    linewidth=2.5,
    label=f'Fleet: {fleet_conformant_percentage:.2f}%',
)
conformance_ax.set_ylim(0, 100)
conformance_ax.set_yticks(range(0, 101, 5))
conformance_ax.set_xlabel('Site ID', fontsize=14)
conformance_ax.set_ylabel(
    'Passive anti-islanding conformant timestamps (%)', fontsize=14
)
conformance_ax.set_xticks(x_positions)
conformance_ax.set_xticklabels(site_labels, rotation=90)

timestamp_ax = conformance_ax.twinx()
timestamp_line, = timestamp_ax.plot(
    x_positions,
    passive_anti_islanding_conformance_plot[
        'num_eligible_timestamps'
    ].to_list(),
    color='darkorange',
    marker='o',
    markersize=4,
    linewidth=2.5,
    label='Eligible timestamps',
)
timestamp_ax.set_ylabel(
    'Number of eligible timestamps', color='darkorange', fontsize=14
)
timestamp_ax.tick_params(axis='y', labelcolor='darkorange')

conformance_ax.set_title(
    'Passive anti-islanding conformance and eligible timestamps by site',
    fontsize=16,
)
conformance_ax.legend(
    handles=[threshold_line, fleet_line, timestamp_line],
    loc='lower right',
)
fig.tight_layout()
plt.show()

num_sites_investigated = (
    passive_anti_islanding_conformance_summary.height
)
num_sites_eligible = passive_anti_islanding_conformance_plot.height
num_sites_conformant = (
    passive_anti_islanding_conformance_plot
    .filter(pl.col('conformant_percentage') >= 90)
    .height
)
conformant_percentage_of_cohort = (
    num_sites_conformant / num_sites_eligible * 100
)

print(f'Number of sites investigated: {num_sites_investigated}')
print(f'Number of sites eligible: {num_sites_eligible}')
print(f'Number of sites conformant: {num_sites_conformant}')
print(
    f'Conformant percentage of cohort: '
    f'{conformant_percentage_of_cohort:.2f}%'
)